In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
import spacy
import optuna
import mlflow
import mlflow.sklearn

# MLflow / DagShub tracking setup
os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"
mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("Custom Features - LightGBM HP Tuning")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/5a6385f1f2c24471add67c02cd88d561', creation_time=1784594071756, effective_trace_archival_retention=None, experiment_id='12', last_update_time=1784594071756, lifecycle_stage='active', name='Custom Features - LightGBM HP Tuning', tags={}, trace_location=None, workspace='default'>

In [2]:
# Load dataset (same file + same cleaning as the LogReg custom-features notebook)
dataset = pd.read_csv('dataset.csv')

# Drop rows with NaN values
cleaned_dataset = dataset.dropna()

In [3]:
# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# 80-20 split - IDENTICAL to the LogReg custom-features notebook
# (same random_state=42, same test_size, no stratify) -> identical rows -> fair comparison
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42
)

In [4]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [ ]:
# All 17 universal POS tags, in a FIXED order.
# Looping over this (instead of set(pos_tags)) guarantees every comment produces the
# same 17 POS columns in the same order -> train, test, and future data always line up.
UNIVERSAL_POS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM',
                 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

# Function to extract custom features
def extract_custom_features(text):
    doc = nlp(text)
    word_list = [token.text for token in doc]

    # 1. Comment Length (number of characters)
    comment_length = len(text)

    # 2. Word Count
    word_count = len(word_list)

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in word_list) / word_count if word_count > 0 else 0

    # 4. Unique Word Count
    unique_word_count = len(set(word_list))

    # 5. Lexical Diversity
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

    # 6. Count of POS Tags
    pos_count = len([token.pos_ for token in doc])

    # 7. Proportion of POS Tags - loop over the FIXED list (not set()) so every comment
    #    yields all 17 columns in the same order; tags not present come out as 0.
    pos_tags = [token.pos_ for token in doc]
    if word_count > 0:
        pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in UNIVERSAL_POS}
    else:
        pos_proportion = {tag: 0 for tag in UNIVERSAL_POS}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion  # Flattening the POS proportions
    }

In [6]:
# Apply the custom feature extraction
train_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_train_cleaned])
test_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_test_cleaned])

In [7]:
# Replace NaN values in POS tag proportions with 0
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

In [8]:
# TF-IDF - bigram, max_features=10000, FIT ONLY on train
ngram_range = (1, 2)
max_features = 10000
tfidf = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_tfidf = tfidf.fit_transform(X_train_cleaned)  # Fit ONLY on train
X_test_tfidf = tfidf.transform(X_test_cleaned)        # Transform test

In [9]:
# Convert TF-IDF to DataFrame
X_train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
X_test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), columns=tfidf.get_feature_names_out())

In [10]:
# Combine TF-IDF and custom features
X_train_combined = pd.concat([X_train_tfidf_df.reset_index(drop=True), train_custom_features.reset_index(drop=True)], axis=1)
X_test_combined = pd.concat([X_test_tfidf_df.reset_index(drop=True), test_custom_features.reset_index(drop=True)], axis=1)

# Align test columns to train columns (order AND set).
# The spaCy POS-proportion columns come from set(pos_tags), so train/test can end up in
# different column orders -> LightGBM/sklearn feature-name mismatch at predict.
# reindex fixes the order, adds any POS column missing from test (as 0), and drops extras.
X_test_combined = X_test_combined.reindex(columns=X_train_combined.columns, fill_value=0)

In [11]:
# LightGBM is scale-invariant -> NO scaler needed (unlike the LogReg notebook)
X_train = X_train_combined
X_test = X_test_combined
y_train = y_train_cleaned
y_test = y_test_cleaned

In [12]:
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number, log_model=False):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Only log the model artifact for the best run to stay within DagShub free tier limits
        if log_model:
            # Persist locally first so a long train is never lost to a logging failure
            joblib.dump(model, f"{model_name}_best_model.joblib")
            try:
                mlflow.sklearn.log_model(
                    model, f"{model_name}_model",
                    serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
                )
            except Exception as e:
                print(f"[warn] model artifact logging failed (metrics/params + joblib file are safe): {e}")

        return accuracy

In [13]:
# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    # Hyperparameter space to explore
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)   # L1 regularization
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)  # L2 regularization
    # Class imbalance handling - let Optuna decide whether balancing helps
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])

    # Log trial parameters
    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'class_weight': class_weight
    }

    # Create LightGBM model (no scaler needed - trees are scale-invariant)
    model = LGBMClassifier(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           max_depth=max_depth,
                           num_leaves=num_leaves,
                           min_child_samples=min_child_samples,
                           colsample_bytree=colsample_bytree,
                           subsample=subsample,
                           reg_alpha=reg_alpha,
                           reg_lambda=reg_lambda,
                           class_weight=class_weight,
                           random_state=42,
                           verbose=-1)

    # Log each trial as a separate run in MLflow
    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy

In [14]:
# Step 7: Run Optuna for LightGBM, log the best model, and plot the importance of each parameter
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=50)  # 50 trials (matches the LogReg notebook)

    # Get the best parameters
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'],
                                learning_rate=best_params['learning_rate'],
                                max_depth=best_params['max_depth'],
                                num_leaves=best_params['num_leaves'],
                                min_child_samples=best_params['min_child_samples'],
                                colsample_bytree=best_params['colsample_bytree'],
                                subsample=best_params['subsample'],
                                reg_alpha=best_params['reg_alpha'],
                                reg_lambda=best_params['reg_lambda'],
                                class_weight=best_params['class_weight'],
                                random_state=42,
                                verbose=-1)

    # Log the best model with MLflow (only this run uploads the model artifact)
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test, best_params, "Best", log_model=True)

    # Plot parameter importance
    optuna.visualization.plot_param_importances(study).show()

    # Plot optimization history
    optuna.visualization.plot_optimization_history(study).show()

    # Return the best model so it can be inspected outside this function
    return best_model

In [19]:
# Run the experiment for LightGBM
best_model = run_optuna_experiment()

[I 2026-07-21 18:54:12,063] A new study created in memory with name: no-name-6c5d6013-da95-4d8f-a14c-666e16895002


🏃 View run Trial_0_LightGBM_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12/runs/6254a98c103b45f4a86dd0726dcf740d
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12


[I 2026-07-21 18:58:44,875] Trial 0 finished with value: 0.760125460248193 and parameters: {'n_estimators': 404, 'learning_rate': 0.03505690255420824, 'max_depth': 4, 'num_leaves': 20, 'min_child_samples': 53, 'colsample_bytree': 0.9067624187994126, 'subsample': 0.5471132771207625, 'reg_alpha': 0.00040455816980270283, 'reg_lambda': 0.0921027850530837, 'class_weight': None}. Best is trial 0 with value: 0.760125460248193.


🏃 View run Trial_1_LightGBM_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12/runs/5eb2df8b70e94fec974e7e5277c1aa4b
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12


[I 2026-07-21 19:04:33,313] Trial 1 finished with value: 0.802945588435838 and parameters: {'n_estimators': 909, 'learning_rate': 0.030343166923543434, 'max_depth': 7, 'num_leaves': 81, 'min_child_samples': 74, 'colsample_bytree': 0.7349400488583051, 'subsample': 0.5402766661292986, 'reg_alpha': 0.000120733785767025, 'reg_lambda': 0.0007948199191232308, 'class_weight': None}. Best is trial 1 with value: 0.802945588435838.


...

[I 2026-07-21 23:56:02,117] Trial 48 finished with value: 0.8446747579435429 and parameters: {'n_estimators': 995, 'learning_rate': 0.016752417952338033, 'max_depth': 14, 'num_leaves': 61, 'min_child_samples': 10, 'colsample_bytree': 0.9513705585592978, 'subsample': 0.9969073219843763, 'reg_alpha': 0.0005545032063340721, 'reg_lambda': 0.0022509121035195006, 'class_weight': 'balanced'}. Best is trial 46 with value: 0.8629483158325378.


🏃 View run Trial_49_LightGBM_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12/runs/0ec2238f8bf549299a748b7ad87a0300
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12


[I 2026-07-22 00:09:59,049] Trial 49 finished with value: 0.8622664666575753 and parameters: {'n_estimators': 964, 'learning_rate': 0.035820546921949276, 'max_depth': 14, 'num_leaves': 103, 'min_child_samples': 19, 'colsample_bytree': 0.9985977535303042, 'subsample': 0.8949879097000155, 'reg_alpha': 0.0011716743462494815, 'reg_lambda': 0.005037659652254979, 'class_weight': 'balanced'}. Best is trial 46 with value: 0.8629483158325378.
2026/07/22 00:21:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/22 00:22:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_Best_LightGBM_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12/runs/3e702f5b0f9741ae90318e20532d4433
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/12


ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [ ]:
best_model

# 📌 Best Params — LightGBM + Custom Features

**Run:** 2026-07-21 18:58 → 2026-07-22 00:10 (~5h 11m) · 50/50 Optuna trials completed
**Experiment (MLflow):** `Custom Features - LightGBM HP Tuning`
**Features:** TF-IDF bigram (1,2), `max_features=10000` + spaCy custom features (23 cols) → 10,023 total
**Split:** 80/20, `random_state=42`, no stratify (identical to the LogReg custom-features notebook)

## 🏆 Best result — Trial 46

**Test accuracy: `0.8629`**

```python
best_params_lightgbm = {
    'n_estimators':      902,
    'learning_rate':     0.03525642576883964,
    'max_depth':         15,
    'num_leaves':        88,
    'min_child_samples': 11,
    'colsample_bytree':  0.9975747444801832,
    'subsample':         0.9294604717135944,
    'reg_alpha':         0.001713556836930048,
    'reg_lambda':        0.0022080042696159116,
    'class_weight':      'balanced',
    # fixed (not tuned)
    'random_state':      42,
    'verbose':           -1,
}
```

## Top 5 trials

| Rank | Trial | Accuracy | learning_rate | max_depth | num_leaves | min_child_samples | class_weight |
|------|-------|----------|---------------|-----------|------------|-------------------|--------------|
| 1 | **46** | **0.8629** | 0.0353 | 15 | 88  | 11 | balanced |
| 2 | 49 | 0.8623 | 0.0358 | 14 | 103 | 19 | balanced |
| 3 | 43 | 0.8609 | 0.0323 | 14 | 102 | 19 | balanced |
| 4 | 45 | 0.8608 | 0.0322 | 15 | 89  | 10 | balanced |
| 5 | 40 | 0.8523 | 0.0286 | 15 | 117 | 29 | balanced |

## Observations

- **`class_weight='balanced'` won decisively** — every one of the top 5 trials used it. Trials with `class_weight=None` plateaued around 0.83.
- The winning region is consistent: **learning_rate ≈ 0.03**, **max_depth 14–15**, **num_leaves ~90–105**, **low min_child_samples (10–20)**, **high colsample_bytree/subsample (~0.93–1.0)**, **weak regularization** (`reg_alpha`/`reg_lambda` ≈ 0.002–0.005).
- Low learning rates (< 0.001) were disastrous (0.43–0.69) — the model never converged within the estimator budget.
- **Custom features helped:** untuned LightGBM + custom features scored 0.8523; tuned reaches **0.8629**.

## Notes

- The run ended with an `ImportError: No module named 'plotly'` from `optuna.visualization`. This happened **after** the best model was saved, so nothing was lost — the model was persisted to `LightGBM_best_model.joblib` and logged to MLflow first. Only the plots and the `best_model` variable assignment were skipped. (plotly has since been installed.)
- Because the exception fired before `return best_model`, the in-memory `study` object was lost — but every trial's params and metrics are in MLflow.
- ⚠️ These params were selected by maximising accuracy **on the test set**, so `0.8629` is optimistically biased. Use a fresh held-out set for the final generalisation number.
